In [1]:
# [목적] 필요한 라이브러리와 API 키 환경을 준비합니다.
# LangSmith 설정은 모델 실행 과정을 추적할 때 사용합니다.
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("Chapter6-OutputParser")


LangSmith 추적을 시작합니다.
[프로젝트명]
Chapter6-OutputParser


In [2]:
# [목적] AI의 쉼표 구분 답변을 Python 목록으로 바꿀 파서를 만듭니다.
# 파서가 요구하는 출력 형식 안내문도 함께 가져옵니다.
from langchain_core.output_parsers import CommaSeparatedListOutputParser
output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

In [3]:
# [목적] 파서가 만든 목록 출력 안내문을 직접 확인합니다.
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [4]:
# [목적] 질문과 출력 형식 안내를 하나의 프롬프트로 만듭니다.
# {subject}는 실행할 때 실제 주제로 바뀝니다.
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

In [5]:
# [목적] 프롬프트 → AI 모델 → 목록 파서를 하나의 실행 흐름으로 연결합니다.
# temperature=0은 답변의 무작위성을 낮춰 결과를 일정하게 만듭니다.
model = ChatOpenAI(temperature=0)

chain = prompt | model | output_parser

In [6]:
# [목적] 주제를 넣어 체인을 실행하고 완성된 Python 목록을 한 번에 받습니다.
# 반환된 목록은 저장하거나 반복 처리할 수 있습니다.
answer = chain.invoke({"subject": "대한민국 관광명소"})

answer

['경복궁', '남산타워', '부산 해운대해수욕장', '제주도 성산일출봉', '경주 불국사temples']

In [7]:
# [목적] stream으로 결과를 조금씩 받아 만들어지는 순서대로 출력합니다.
# 긴 답변을 실시간으로 보여주고 싶을 때 사용하는 방식입니다.
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)

['경복궁']
['남산타워']
['부산 해운대해수욕장']
['제주도 성산일출봉']
['경주 불국사temples']
